# 01. Exploratory Data Analysis (EDA) - California Housing Dataset

This notebook provides an in-depth exploratory data analysis of the California Housing dataset to understand feature distributions, identify correlations, inspect potential outliers, and establish foundational insights before predictive modeling.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing

# Set styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


### 1. Data Loading

We load the California Housing dataset using `sklearn.datasets.fetch_california_housing`. We ensure the data is loaded into a Pandas DataFrame for structured exploration.


In [ ]:
data_dir = os.path.abspath("./data")
housing = fetch_california_housing(data_home=data_dir, as_frame=True)
df = housing.frame

print(f"Dataset Shape: {df.shape[0]} samples, {df.shape[1]} columns")
df.head()


### 2. Schema and Statistical Inspection

Let's inspect the data types, non-null counts, and summary statistics across all features using `df.info()` and `df.describe()`.


In [ ]:
df.info()

In [ ]:
# Missing value check
missing_summary = df.isnull().sum()
print("Missing values per column:")
print(missing_summary)


In [ ]:
# Summary statistics
df.describe().T


**Observations from summary statistics:**
- **No Missing Values**: Every feature contains complete entries (20,640 non-null values).
- **Target Variable (`MedHouseVal`)**: Median house value is expressed in hundreds of thousands of dollars ($100,000). The mean is ~2.07 ($207,000), with a standard deviation of 1.15. The maximum value caps at 5.00001 ($500,000), indicating artificial top-coding (censoring).
- **Income (`MedInc`)**: Median income has a median of 3.53 (~$35,300) and max of 15.0.
- **Outliers / Extremes**: Features such as `AveRooms`, `AveBedrms`, and `AveOccup` display high maximums compared to their 75th percentiles, reflecting skewed distributions or dense localized clusters.


### 3. Target Variable Distribution (`MedHouseVal`)

Visualizing the distribution of the continuous target variable to assess normality, skewness, and potential capping.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with KDE
sns.histplot(df['MedHouseVal'], kde=True, bins=50, color='royalblue', ax=axes[0])
axes[0].set_title("Distribution of Median House Value (MedHouseVal)")
axes[0].set_xlabel("Median House Value ($100k)")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df['MedHouseVal'], color='skyblue', ax=axes[1])
axes[1].set_title("Boxplot of Median House Value")
axes[1].set_xlabel("Median House Value ($100k)")

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/target_distribution.png", dpi=200)
plt.show()


**Analysis of Target Distribution:**
- The distribution is right-skewed with a peak around $150,000 - $200,000.
- Notice the spike at $5.0 ($500,000). Houses valued at $500,000 or more were capped during survey collection. Machine learning models should be evaluated carefully around this boundary.


### 4. Correlation Analysis

A correlation heatmap allows us to quantify linear associations between the explanatory features and `MedHouseVal`.


In [ ]:
plt.figure(figsize=(10, 8))
correlation_matrix = df.corr()

sns.heatmap(
    correlation_matrix, 
    annot=True, 
    fmt=".2f", 
    cmap="coolwarm", 
    vmin=-1, 
    vmax=1, 
    linewidths=0.5
)
plt.title("Correlation Matrix of California Housing Features", fontsize=14, pad=12)
plt.savefig("figures/correlation_heatmap.png", dpi=200)
plt.show()


**Correlation Insights:**
- **Median Income (`MedInc`)** has the strongest positive linear correlation with `MedHouseVal` ($r = 0.69$). As household income increases, property value increases proportionally.
- **Latitude & Longitude** display moderate negative correlation with each other ($-0.92$), tracing California's northwest-to-southeast geographic slant.
- **Rooms and Bedrooms** (`AveRooms` and `AveBedrms`) have a strong collinearity ($r = 0.85$). Tree-based models (Random Forest, XGBoost) handle this collinearity natively, whereas regularized models or baseline linear regression can see variance inflation.


### 5. Deep-Dive: Median Income vs Median House Value

Because `MedInc` is our most predictive univariate feature, let's plot a hexbin / scatter plot to visualize the density.


In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df, 
    x="MedInc", 
    y="MedHouseVal", 
    alpha=0.2, 
    color="darkblue"
)
plt.title("Median Income vs. Median House Value", fontsize=13)
plt.xlabel("Median Income ($10k units)")
plt.ylabel("Median House Value ($100k units)")
plt.savefig("figures/medinc_vs_medhouseval.png", dpi=200)
plt.show()


### 6. Geographic Distribution (Latitude vs. Longitude)

California housing values are heavily driven by coastal geography (Bay Area and Los Angeles coastal regions).


In [ ]:
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    df['Longitude'], 
    df['Latitude'], 
    c=df['MedHouseVal'], 
    cmap='viridis', 
    alpha=0.4, 
    s=df['Population']/100
)
plt.colorbar(scatter, label="Median House Value ($100k)")
plt.title("California Housing Prices by Geographic Location (Bubble size: Population)", fontsize=13)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.savefig("figures/geographic_distribution.png", dpi=200)
plt.show()


## EDA Summary & Conclusions

1. **Dataset Integrity**: 20,640 samples, 8 predictive numerical features, and zero missing values.
2. **Key Predictors**: `MedInc` (Median Income) is by far the strongest single driver of property value ($r = 0.69$).
3. **Geographic Clustering**: High property values cluster tightly along the coast and major metropolitan centers (San Francisco Bay Area, Coastal Southern California).
4. **Target Censoring**: Values above $5.0 ($500,000) are capped, creating an artificial ceiling that non-linear ensemble models (Random Forest and XGBoost) can navigate better than standard linear regressors.
5. **Next Steps**: Proceed to preprocessing (StandardScaler) and benchmark Linear Regression against Random Forest and XGBoost in `02_Modeling.ipynb` / `train.py`.
